# Satellite Image Land-Cover Classification System - Google Colab Setup

This notebook allows you to mount Google Drive, install dependencies, authenticate Earth Engine, run the pipeline on free GPU compute, and host the Streamlit dashboard using localtunnel.

### Project Directory Structure in Google Drive

Upload the project files from your local computer to your Google Drive in this structure:
```
My Drive/
└── Satellite_Image_Project/
    ├── app.py
    ├── app_api.py
    ├── run_pipeline.py
    └── src/
        ├── __init__.py
        ├── analysis/
        │   ├── __init__.py
        │   ├── ndvi_anomaly.py
        │   └── timeseries_trends.py
        ├── data/
        │   ├── __init__.py
        │   ├── dataset.py
        │   ├── download_ee.py
        │   ├── download_quickstart.py
        │   ├── download_sar.py
        │   ├── fusion_dataset.py
        │   ├── raster_utils.py
        │   ├── sar_preprocess.py
        │   └── timeseries.py
        ├── models/
        │   ├── __init__.py
        │   ├── attention_fusion.py
        │   ├── benchmark.py
        │   ├── sar_translator.py
        │   ├── segformer_module.py
        │   └── train_ssl.py
        ├── training/
        │   ├── __init__.py
        │   └── augmentations.py
        └── vis/
            ├── __init__.py
            ├── map_export.py
            └── report_exporter.py
```

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### ⚠️ troubleshooting: Google Drive Mount Failure

If `drive.mount` fails with **`MessageError: Error: credential propagation was unsuccessful`**, it is usually because:
1. You are signed into **multiple Google accounts** in your browser. *Fix:* Sign out of other accounts, or open this notebook in an **Incognito/Private window**.
2. Third-party cookies or cross-site tracking are disabled in your browser settings. *Fix:* Enable them for `colab.research.google.com`.

**Alternative Fallback:** If you cannot get the Drive mount to work, run the cell below to upload your project directory as a `.zip` file directly to the Colab local file system.

In [ ]:
# RUN THIS ONLY IF GOOGLE DRIVE MOUNT FAILS
# 1. Create a zip of your project files on your computer (containing app.py, run_pipeline.py, src/, etc.)
# 2. Run this cell to upload and extract it directly into Colab's temp storage
import os
import zipfile
from google.colab import files

if not os.path.exists('/content/drive/MyDrive/Satellite_Image_Project'):
    print("Drive not mounted. Let's upload the project files directly as a ZIP archive:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        print(f'Extracting uploaded file "{fn}"...')
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✅ Extraction complete! Current files in workspace:")
        print(os.listdir('.'))
else:
    print("✅ Google Drive is already mounted. Skipping direct upload.")

In [ ]:
# 2. Install Required Dependencies on Google Colab (Silent Install)
!pip install -q \
    pytorch-lightning \
    segmentation-models-pytorch \
    geemap \
    folium \
    streamlit \
    streamlit-folium \
    transformers \
    albumentations \
    rasterio \
    opencv-python \
    scipy \
    pandas \
    fastapi \
    uvicorn

In [ ]:
# 3. Change Directory to the Google Drive folder
import os
os.chdir('/content/drive/MyDrive/Satellite_Image_Project')
print("Current Working Directory:", os.getcwd())

In [ ]:
%env GEE_PROJECT=local-cogency-474306-h1

In [ ]:
# 4. Authenticate Google Earth Engine
import ee
ee.Authenticate()

In [ ]:
# 5. Run the Training & Processing Pipeline

# Option A: Run baseline training & evaluation
# Model can be: "unet", "segformer", "both", or "cross_attention_unet"
!python run_pipeline.py --mode gee --model both --fusion

# Option B: Run Self-Supervised NT-Xent contrastive training (SimCLR) before main model
# !python run_pipeline.py --mode gee --model cross_attention_unet --train_ssl

# Option C: Train the Pix2Pix SAR-to-Optical translator (conditional GAN)
# !python run_pipeline.py --mode gee --model cross_attention_unet --train_translator

## 6. Expose the Streamlit Dashboard (Optional)

1. Start the Streamlit app in the background.
2. Fetch your Colab public IP (needed as password for localtunnel).
3. Expose the local port 8501 using localtunnel.

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

# Ensure plots look sharp and are displayed inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

# Paths (using relative paths since you changed directory into the project folder)
ROOT = Path(".")
OUTPUT_DIR = ROOT / "outputs"
MAP_DIR = OUTPUT_DIR / "maps"
REPORT_DIR = OUTPUT_DIR / "reports"

# Class and color mapping
CLASS_NAMES = [
    "Urban", "Forest", "Cropland", "Grassland", "Bare Soil",
    "Wetlands", "Water", "Snow", "Shrubland", "Clouds"
]
CLASS_COLORS = [
    "#FF0000", "#006400", "#FFD700", "#7CFC00", "#D2B48C",
    "#00CED1", "#0000FF", "#FFFFFF", "#8B4513", "#808080"
]

def show_image(path, title=None, figsize=(12, 8)):
    """Helper to display a saved plot image directly in the notebook."""
    if Path(path).exists():
        plt.figure(figsize=figsize)
        img = Image.open(path)
        plt.imshow(img)
        plt.axis('off')
        if title:
            plt.title(title, pad=15)
        plt.show()
    else:
        print(f"ℹ️ Image not found (may need to run that pipeline mode first): {path}")

print("✅ Visualization config loaded successfully.")

In [ ]:
metrics_file = REPORT_DIR / "metrics.json"

if metrics_file.exists():
    with open(metrics_file, "r") as f:
        metrics = json.load(f)

    # 1. Print Summary Table
    print("=" * 45)
    print("📊 OVERALL PIPELINE EVALUATION METRICS")
    print("=" * 45)
    print(f"Overall Pixel Accuracy : {metrics['overall_accuracy'] * 100:.2f}%")
    print(f"Mean IoU (mIoU)        : {metrics['mean_iou'] * 100:.2f}%")
    if "mean_confidence" in metrics:
        print(f"Mean Model Confidence  : {metrics['mean_confidence'] * 100:.2f}%")
    print("=" * 45)

    # 2. Plot Per-Class IoU
    per_class_iou = metrics["per_class_iou"]
    df_iou = pd.DataFrame({
        "Class": list(per_class_iou.keys()),
        "IoU (%)": [val * 100 for val in per_class_iou.values()]
    }).sort_values(by="IoU (%)", ascending=True)

    colors_dict = dict(zip(CLASS_NAMES, CLASS_COLORS))
    bar_colors = [colors_dict.get(cls, "#808080") for cls in df_iou["Class"]]

    plt.figure(figsize=(10, 6))
    bars = plt.barh(df_iou["Class"], df_iou["IoU (%)"], color=bar_colors, edgecolor='grey', alpha=0.85)

    # Add values on the bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, bar.get_y() + bar.get_height()/2,
                 f'{width:.1f}%',
                 va='center', ha='left', fontweight='bold')

    plt.title("Per-Class Intersection-over-Union (IoU)")
    plt.xlabel("IoU (%)")
    plt.xlim(0, 110)
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("❌ No metrics file found. Please run the pipeline to generate results.")

In [ ]:
# 1. Sample Predictions (RGB | Ground Truth | Prediction)
show_image(MAP_DIR / "sample_predictions.png", "🖼️ Sentinel-2 RGB | Ground Truth | Model Prediction")

# 2. Full Stitched Scene (Quickstart mode)
show_image(MAP_DIR / "full_stitched_scene.png", "🖼️ Full Scene Reassembled (Test Grid)")

# 3. Model Uncertainty Maps
show_image(
    MAP_DIR / "uncertainty_maps.png",
    "🔮 Model Confidence & Uncertainty Map (Green = High Confidence, Red = Uncertain)"
)

In [ ]:
# 1. Confusion Matrix
show_image(MAP_DIR / "confusion_matrix.png", "📉 Model Classification Confusion Matrix", figsize=(10, 8))

# 2. Training History Curves
log_dir = Path("outputs/lightning_logs")
if log_dir.exists():
    versions = sorted(log_dir.glob("version_*"), key=os.path.getmtime, reverse=True)
    if versions:
        metrics_csv = versions[0] / "metrics.csv"
        if metrics_csv.exists():
            try:
                df_log = pd.read_csv(metrics_csv)

                # Check if epoch exists and forward-fill/backward-fill to align step-level logs
                if "epoch" in df_log.columns:
                    df_log["epoch"] = df_log["epoch"].ffill().bfill()
                    df_epoch = df_log.groupby("epoch").mean().reset_index()
                else:
                    df_epoch = df_log

                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

                # Plot Loss Curve
                loss_cols = [c for c in ["train_loss", "val_loss"] if c in df_epoch.columns]
                if loss_cols and "epoch" in df_epoch.columns:
                    for col in loss_cols:
                        ax1.plot(df_epoch["epoch"], df_epoch[col], marker='o', label=col)
                    ax1.set_title("Training & Validation Loss")
                    ax1.set_xlabel("Epoch")
                    ax1.set_ylabel("Loss")
                    ax1.grid(True, linestyle='--', alpha=0.6)
                    ax1.legend()
                else:
                    ax1.text(0.5, 0.5, "No loss metrics found in log", ha='center', va='center')

                # Plot Learning Rate
                lr_cols = [c for c in df_epoch.columns if "lr" in c.lower()]
                if lr_cols and "epoch" in df_epoch.columns:
                    for col in lr_cols:
                        ax2.plot(df_epoch["epoch"], df_epoch[col], marker='s', color='orange', label="Learning Rate")
                    ax2.set_title("Learning Rate Schedule")
                    ax2.set_xlabel("Epoch")
                    ax2.set_ylabel("LR")
                    ax2.grid(True, linestyle='--', alpha=0.6)
                    ax2.legend()
                else:
                    ax2.text(0.5, 0.5, "No learning rate recorded", ha='center', va='center')

                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f"⚠️ Could not extract training logs: {e}")
        else:
            print("ℹ️ metrics.csv not found in the latest lightning log version.")
    else:
        print("ℹ️ No lightning logs versions found.")
else:
    print("ℹ️ Lightning logs folder not found.")

In [ ]:
# 1. Change Detection Visual Maps
show_image(MAP_DIR / "change_detection_maps.png", "🔄 Temporal Change Detection (T1 vs T2)")

# 2. Transition Matrix Area (Hectares)
trans_file = REPORT_DIR / "transition_area_ha.csv"
if trans_file.exists():
    df_trans = pd.read_csv(trans_file, index_col=0)
    print("\n📈 Transition Matrix Area (Hectares):")
    # Style the pandas DataFrame with a beautiful color gradient
    display(df_trans.style.background_gradient(cmap="YlOrRd").format("{:.2f}"))
else:
    print("ℹ️ Change detection transition matrix not found.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipydisplay

# 1. NDVI Stats & Curves
show_image(MAP_DIR / "ndvi_curve.png", "🌿 Monthly NDVI Time-Series Profile")

stats_path = REPORT_DIR / "ndvi_stats.json"
if stats_path.exists():
    with open(stats_path, "r") as f:
        stats = json.load(f)

    MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    print("=" * 45)
    print("🌱 VEGETATION HEALTH METRICS")
    print("=" * 45)
    print(f"Peak Greenness Month   : {MONTH_NAMES[stats['peak_month']]}")
    print(f"Trough (Lowest) Month  : {MONTH_NAMES[stats['trough_month']]}")
    print(f"Anomalous Land Pixels  : {stats['pct_anomalous']:.2f}%")
    print("=" * 45)

show_image(MAP_DIR / "ndvi_anomaly_map.png", "⚠️ Vegetation Anomaly Map (>2.0 standard dev.)")

# 2. Interactive Month Slider for NDVI frames
ts_dir = MAP_DIR / "timeseries"
if ts_dir.exists():
    print("\n🌿 Drag the slider to animate/scrub through monthly NDVI maps:")

    def view_month(month_idx):
        frame_path = ts_dir / f"ndvi_frame_{month_idx:02d}.png"
        if frame_path.exists():
            img = Image.open(frame_path)
            plt.figure(figsize=(10, 6))
            plt.imshow(img)
            plt.title(f"NDVI Heatmap - {MONTH_NAMES[month_idx]} (Green = Vegetation, Red = Bare/Urban)")
            plt.axis('off')
            plt.show()
        else:
            print(f"Frame for month index {month_idx} not found.")

    widgets.interact(view_month, month_idx=widgets.IntSlider(min=0, max=11, step=1, value=6, description='Month:'))
else:
    print("ℹ️ Monthly timeseries NDVI frames are not available.")

In [ ]:
# 3. NDVI Trend Analysis (Slope Map and Greening/Degradation Stats)
show_image(MAP_DIR / "ndvi_trends.png", "📈 12-Month NDVI Trend Slope Map (Green = Greening, Red = Degrading)")

trends_json = REPORT_DIR / "ndvi_trends.json"
if trends_json.exists():
    with open(trends_json, "r") as f:
        trends = json.load(f)
    print("=" * 45)
    print("📈 MULTI-TEMPORAL NDVI TREND STATISTICS")
    print("=" * 45)
    print(f"Vegetation Greening Rate   : {trends['pct_greening']:.2f}% of pixels")
    print(f"Vegetation Degradation Rate: {trends['pct_degrading']:.2f}% of pixels")
    print(f"Stable Vegetation Rate     : {trends['pct_stable']:.2f}% of pixels")
    print(f"Mean Monthly NDVI Slope    : {trends['mean_slope']:.4f}")
    print("=" * 45)
else:
    print("ℹ️ NDVI trends statistics not found. Ensure you ran the full pipeline.")

In [ ]:
# 1. Fusion vs Optical Plot
show_image(MAP_DIR / "fusion_vs_optical.png", "📡 Optical vs SAR Fusion Segmentation Comparison")

# 2. Cloud Recovery Experiment Plot
show_image(
    MAP_DIR / "cloud_recovery.png",
    "☁️ Cloud Occlusion Recovery (SAR radar penetrating cloud cover)"
)

# 3. Print Fusion Metrics
fusion_json = REPORT_DIR / "fusion_results.json"
if fusion_json.exists():
    with open(fusion_json) as f:
        fusion_data = json.load(f)

    if len(fusion_data) >= 2:
        df_fusion = pd.DataFrame(fusion_data)
        print("\n🏆 SAR Fusion Performance Comparison:")
        display(df_fusion[["model", "overall_accuracy", "mean_iou", "train_time_sec"]])

        opt_miou = fusion_data[0]["mean_iou"]
        fuse_miou = fusion_data[1]["mean_iou"]
        print(f"\n✨ SAR Fusion mIoU improvement: {((fuse_miou - opt_miou) * 100):+.2f}%")
else:
    print("ℹ️ SAR Fusion evaluation results not found. Run the pipeline with '--fusion'.")

In [ ]:
import sys
# Make sure the project src dir is in system path for imports
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

try:
    from vis.map_export import build_folium_map
except ImportError:
    from src.vis.map_export import build_folium_map

overlay_path = MAP_DIR / "segmentation_overlay.png"
bounds_path = REPORT_DIR / "map_bounds.json"

if overlay_path.exists() and bounds_path.exists():
    with open(bounds_path, "r") as f:
        bounds = json.load(f)

    # Build and render the folium map directly in the notebook output cell
    m = build_folium_map(str(overlay_path), tuple(bounds))
    print("🗺️ Rendering Interactive GIS Map overlay (toggle layer in top-right):")
    display(m)
else:
    print("ℹ️ Folium Map overlay files not found. Run the pipeline in Quickstart mode first.")

## 8. View Generated Project Summary Report

The pipeline automatically compiles evaluation metrics, transition matrix tables, and NDVI vegetation health stats into a detailed markdown report.

In [ ]:
summary_md = REPORT_DIR / "project_summary_report.md"
if summary_md.exists():
    with open(summary_md, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("ℹ️ Project summary report not found. Ensure report generation succeeded.")

In [ ]:
# Start Streamlit in the background
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &

In [ ]:
# Get public IP for localtunnel password
!curl ipv4.icanhazip.com

In [ ]:
# Expose Streamlit
!npx localtunnel --port 8501

## 9. Expose the FastAPI GIS Map Server (Optional)

The FastAPI dynamic GIS map server (`app_api.py`) serves Web Map Service (WMS/TMS) XYZ tiles from the reprojected final GeoTIFF. You can run it in the background and expose it using localtunnel.

In [ ]:
# Start FastAPI GIS Map Server in the background
!nohup uvicorn app_api:app --host 0.0.0.0 --port 8000 &

In [ ]:
# Expose the FastAPI GIS Map Server public URL
!npx localtunnel --port 8000